In [4]:
using Pkg

Pkg.activate("../../../../")
using RigidBodyDynamics
using MeshCatMechanisms
using MeshCat
using SymPy

  Activating project at `~/planning_on_biped_robot/Code/MPCBipedRobot`


# Visualiser
The purpose of this notebook is to visualise the built urdfs with onshape, and identify potential issues.
There is a separate environment for this notebook because there are compatibility issues between MeshCatMechanisms and ModelingToolkit for instance.

## 1. Load mechanism

In [100]:
urdf_path = "robot.urdf"
robot = parse_urdf(Float64, urdf_path)
remove_fixed_tree_joints!(robot)
joints(robot)


6-element Vector{Joint{Float64, JT} where JT<:JointType{Float64}}:
 Joint "hip_left": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "hip_right": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "knee_left": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "knee_right": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "ankle_left": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "ankle_right": Revolute joint with axis [0.0, 1.0, 0.0]

## 2. Visualisation

In [101]:
vis = MechanismVisualizer(robot, URDFVisuals(urdf_path));
robot_bodies = RigidBodyDynamics.bodies(robot)
for body in robot_bodies
    frame = RigidBodyDynamics.default_frame(body)
    setelement!(vis, frame)
end

┌ Info: Listening on: 127.0.0.1:8734, thread id: 1
└ @ HTTP.Servers /home/brix/.julia/packages/HTTP/4AUPl/src/Servers.jl:382
┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
│ http://127.0.0.1:8734
└ @ MeshCat /home/brix/.julia/packages/MeshCat/9QrxD/src/visualizer.jl:43


## 3. Further analysis

### Define and set the state

In [102]:
state = MechanismState(robot)

MechanismState{Float64, Float64, Float64, …}(…)

In [103]:
hip_left, hip_right, knee_left, knee_right, ankle_left, ankle_right = joints(robot)


6-element Vector{Joint{Float64, JT} where JT<:JointType{Float64}}:
 Joint "hip_left": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "hip_right": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "knee_left": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "knee_right": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "ankle_left": Revolute joint with axis [0.0, 1.0, 0.0]
 Joint "ankle_right": Revolute joint with axis [0.0, 1.0, 0.0]

In [105]:
set_configuration!(state, hip_right, pi/2)
set_configuration!(state, knee_right, pi/4)
set_configuration!(state, ankle_right, pi/3)
set_configuration!(state, hip_left, pi/2)
set_configuration!(state, knee_left, pi/4)
set_configuration!(state, ankle_left, pi/3)

# set_configuration!(state, hip_right, pi/10)
# set_configuration!(state, knee_right, pi/10)
# set_configuration!(state, hip_left, pi/10)
# set_configuration!(state, knee_left, pi/10)

# set_configuration!(state, hip_right, 0)
# set_configuration!(state, knee_right, 0)
# set_configuration!(state, hip_left, 0)
# set_configuration!(state, knee_left, 0)

# set_configuration!(state, foot, 0)

# set_velocity!(state, hip_right, 10)
# set_velocity!(state, knee_right, 0)
# set_velocity!(state, hip_left, 0)
# set_velocity!(state, knee_left, 0)
zero_velocity!(state)
# **Important**: a `MechanismState` contains cache variables that depend on the configurations and velocities of the joints. These need to be invalidated when the configurations and velocities are changed. To do this, call
setdirty!(state)

q = configuration(state)
v = velocity(state)
print("q=$q\nv=$v")

q=[1.5707963267948966, 1.5707963267948966, 0.7853981633974483, 0.7853981633974483, 1.0471975511965976, 1.0471975511965976]
v=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

How much does the robot weigth and where is its com ? 


In [104]:
println("weight: ", mass(robot), "kg")
println(center_of_mass(state))

println(bodies(robot))
for body in bodies(robot)[2:end]
    inertia = spatial_inertia(body)
    mass = inertia.mass
    moment = inertia.moment
    println("Body: $(body.name), Mass: $mass kg, Moment Inertia: $moment ")
end 

world = bodies(robot)[1]
item = RigidBody{Float64}[]
push!(item, world)
for body in bodies(robot)[2:end]  # Skip the root
    temp_item = copy(item)
    push!(temp_item, body)
    com = center_of_mass(state, temp_item)
    println("CoM of $(body) in world frame: $com")
end


weight: 0.681476kg
Point3D in "world": [1.3374354902230843e-5, 0.04999608266651114, -0.24108626581919243]
RigidBody{Float64}[RigidBody: "world", RigidBody: "leg_top", RigidBody: "leg_top_2", RigidBody: "leg_bottom", RigidBody: "leg_bottom_2", RigidBody: "feet", RigidBody: "feet_2"]
Body: leg_top, Mass: 0.20726 kg, Moment Inertia: [0.0046470222031809475 -1.2555069649436893e-6 1.6820283677298096e-7; -1.2555069649436893e-6 0.0037621343690821276 0.0005301013246453178; 1.68202836772981e-7 0.0005301013246453178 0.0009140667884729959] 
Body: leg_top_2, Mass: 0.20726 kg, Moment Inertia: [0.004647024072401218 -3.2609940209357196e-8 1.7248877738964796e-7; -3.2609940209357196e-8 0.0037621324998618573 -0.000530101321398944; 1.7248877738964798e-7 -0.000530101321398944 0.0009140667884729959] 
Body: leg_bottom, Mass: 0.112193 kg, Moment Inertia: [0.0012481237618943886 -4.90215057806438e-7 4.75315267699105e-8; -4.90215057806438e-7 0.0009091677858767586 0.0001896933568461592; 4.753152676991049e-8 0.000

Now let's compute the endEffector position


In [106]:
endEffector = ("feet" , "feet_2")
for effector in endEffector
    foot_link = findbody(robot, "$effector")
    frame = default_frame(foot_link)
    tf_world_to_body = transform_to_root(state, frame)
    println(tf_world_to_body)
end 

Transform3D from "after_ankle_left" to "world":
rotation: 2.879793265790644 rad about [2.451841184134177e-63, -1.0, -0.0], translation: [-0.3228795326424981, 0.07599, 0.08136943743189798]
Transform3D from "after_ankle_right" to "world":
rotation: 2.879793265790644 rad about [2.451841184134177e-63, -1.0, -0.0], translation: [-0.3228795326425047, 0.023989999999999994, 0.0813694374318914]


### Simulation

In [107]:
ts, qs, vs = simulate(state, 1., Δt = 1e-3);

In [108]:
mvis = MechanismVisualizer(robot, URDFVisuals(urdf_path))
animation = Animation(mvis, ts, qs)
setanimation!(mvis, animation)

# Create a MechanismVisualizer and visualize
MeshCatMechanisms.animate(mvis, ts, qs; realtimerate = 1.)

┌ Info: Listening on: 127.0.0.1:8735, thread id: 1
└ @ HTTP.Servers /home/brix/.julia/packages/HTTP/4AUPl/src/Servers.jl:382
┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
│ http://127.0.0.1:8735
└ @ MeshCat /home/brix/.julia/packages/MeshCat/9QrxD/src/visualizer.jl:43


Let's get the SymPy dynamics now


In [11]:
# Problem size (Number of joints)
n = length(configuration(state)) 

# Sympy type 
T = eltype(SymPy.symbols("_"))

# Create temporary state structure
temp_state = MechanismState{T}(robot) 

# Generalized coordinates
qsym = SymPy.symbols(["qsym[$i]" for i in 1:n], real=true)   
q̇sym = SymPy.symbols(["q̇sym[$i]" for i in 1:n], real=true)
q̈sym = SymPy.symbols(["q̈sym[$i]" for i in 1:n], real=true)


# set the actual state of the robot
set_configuration!(temp_state, qsym)
set_velocity!(temp_state, q̇sym)
setdirty!(temp_state)

# Get base/root body (assuming it is named "base_link")
bodies_list = bodies(robot)
base_link = first(bodies_list)

# Compute system dynamics, Ensure type compatibility
M   = Matrix{T}(mass_matrix(temp_state))  
N   = Vector{T}(RigidBodyDynamics.dynamics_bias(temp_state))  
CoM =  center_of_mass(temp_state)

# Simplify 
M    = SymPy.simplify(M)
Mf   = SymPy.lambdify(M, qsym)    
N    = SymPy.simplify(N)
Nf   = SymPy.lambdify(N, vcat(qsym, q̇sym))
CoM  = SymPy.simplify(CoM)
CoMf = SymPy.lambdify(collect(CoM.v), qsym)  

#157 (generic function with 1 method)

In [12]:
bodies(robot)

7-element Vector{RigidBody{Float64}}:
 RigidBody: "world"
 RigidBody: "leg_top"
 RigidBody: "leg_top_2"
 RigidBody: "leg_bottom"
 RigidBody: "leg_bottom_2"
 RigidBody: "feet"
 RigidBody: "feet_2"